In [13]:
# 1. Instalar dependências necessárias e forçar a atualização do torchao
!pip -q install diffusers transformers accelerate torchmetrics matplotlib huggingface_hub peft
!pip -q install --upgrade torchao

from huggingface_hub import login

import torch
import numpy as np
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline
from transformers import CLIPProcessor, CLIPModel

# Configurações Iniciais
base_model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
lora_model_id = "wesleymqsss/lora-estilo-arquitetura-brasilia-r4"
seed = 42
generator = torch.Generator(device="cuda").manual_seed(seed)

# Inicializar modelo e processador CLIP direto pelo Transformers para evitar o bug
print("Carregando modelo CLIP para métricas...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to("cuda")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

# Nova função de cálculo que contorna o erro e gera a mesma nota da torchmetrics
def calcular_clip(pil_img, prompt):
    inputs = clip_processor(text=[prompt], images=pil_img, return_tensors="pt", padding=True).to("cuda")

    with torch.no_grad():
        outputs = clip_model(**inputs)

    # Extrai os tensores puros e normaliza
    image_embeds = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
    text_embeds = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)

    # O CLIPScore original da torchmetrics é a similaridade de cosseno multiplicada por 100
    clip_score = (image_embeds @ text_embeds.T).item() * 100

    # torchmetrics retorna 0 para valores negativos
    return max(clip_score, 0.0)

# Carregar Modelo Base
print("Carregando modelo base Stable Diffusion...")
pipe = StableDiffusionPipeline.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16
).to("cuda")

# Defina seus 6 prompts de teste
prompts = [
    "estilo_arquitetura_brasilia, um museu moderno com rampas curvas de concreto",
    "estilo_arquitetura_brasilia, uma catedral com colunas apontando para o céu azul",
    "estilo_arquitetura_brasilia, um palácio governamental com um grande espelho d'água",
    "estilo_arquitetura_brasilia, pilotis de concreto sustentando um bloco residencial longo",
    "estilo_arquitetura_brasilia, vista noturna da esplanada com monumentos iluminados",
    "estilo_arquitetura_brasilia, um teatro de formato piramidal em concreto aparente"
]

imagens_base = []
imagens_lora = []
scores_base = []
scores_lora = []

print("Gerando imagens com o Modelo Base...")
for p in prompts:
    img = pipe(p, generator=torch.Generator(device="cuda").manual_seed(seed)).images[0]
    imagens_base.append(img)
    scores_base.append(calcular_clip(img, p))

# Carregar o seu LoRA direto da pasta do Colab (à prova de falhas!)
print("Injetando pesos do LoRA...")
pipe.load_lora_weights(lora_model_id)

print("Gerando imagens com o Modelo + LoRA...")
for p in prompts:
    img = pipe(p, generator=torch.Generator(device="cuda").manual_seed(seed)).images[0]
    imagens_lora.append(img)
    scores_lora.append(calcular_clip(img, p))

# Resultados do CLIPScore (Etapa 2)
print("\n--- RESULTADOS CLIPScore ---")
print(f"Média Modelo Base: {np.mean(scores_base):.2f}")
print(f"Média Modelo + LoRA: {np.mean(scores_lora):.2f}")

# Montar a Grade 6x2 (Etapa 1)
fig, axes = plt.subplots(6, 2, figsize=(10, 24))
fig.suptitle("Comparação: Modelo Base vs. Modelo + LoRA", fontsize=16)

for i in range(6):
    # Coluna 1: Base
    axes[i, 0].imshow(imagens_base[i])
    axes[i, 0].set_title(f"Base\nCLIP: {scores_base[i]:.2f}")
    axes[i, 0].axis('off')

    # Coluna 2: LoRA
    axes[i, 1].imshow(imagens_lora[i])
    axes[i, 1].set_title(f"LoRA\nCLIP: {scores_lora[i]:.2f}")
    axes[i, 1].axis('off')

    # Adicionar o prompt na lateral
    axes[i, 0].text(-0.1, 0.5, f"Prompt {i+1}", transform=axes[i, 0].transAxes,
                    fontsize=12, va='center', ha='right', rotation=90)

plt.tight_layout(rect=[0.05, 0, 1, 0.98])
plt.savefig("grade_comparativa.png")
plt.show()
print("Grade salva como 'grade_comparativa.png'. Você pode baixá-la para o relatório!")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import os
import shutil
from google.colab import files

# 1. Criar uma pasta para organizar as imagens
pasta_saida = "imagens_geradas_avaliacao"
os.makedirs(pasta_saida, exist_ok=True)

print("Salvando imagens na pasta do Colab...")

# 2. Salvar as imagens individuais na pasta
for i in range(6):
    # Salvar imagens do modelo base
    nome_base = f"{pasta_saida}/prompt_{i+1}_modelo_base.png"
    imagens_base[i].save(nome_base)

    # Salvar imagens do modelo com LoRA
    nome_lora = f"{pasta_saida}/prompt_{i+1}_modelo_lora.png"
    imagens_lora[i].save(nome_lora)

# 3. Salvar também a grade comparativa lá dentro (caso o matplotlib já tenha gerado antes)
# Recriamos a imagem da grade rapidamente ou apenas copiamos o arquivo já gerado
if os.path.exists("grade_comparativa.png"):
    shutil.copy("grade_comparativa.png", f"{pasta_saida}/grade_comparativa_completa.png")

# 4. Compactar a pasta inteira em um arquivo .zip
print("Compactando arquivos...")
shutil.make_archive(pasta_saida, 'zip', pasta_saida)

# 5. Fazer o download automático do .zip para o seu PC
print("Iniciando o download para o seu computador...")
files.download(f"{pasta_saida}.zip")

Salvando imagens na pasta do Colab...
Compactando arquivos...
Iniciando o download para o seu computador...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import os
import matplotlib.pyplot as plt
from google.colab import files
import torch

# 1. As 10 legendas exatas do seu dataset
prompts_overfitting = [
    "estilo_arquitetura_brasilia, vista panorâmica do Teatro Nacional de Brasília Claúdio Santoro exibindo toda a arquitetura do teatro em formato de pirâmide sob um céu azul limpo plano aberto",
    "estilo_arquitetura_brasilia, perspectiva ampla do Memorial JK em Brasília do primeiro plano na parte inferior há uma extensão de grama verde bem aparada que ocupa toda a largura da foto logo atrás do gramado a grande estrutura horizontal do memorial, revestida em mármore branco brilhante nesta imagem é possível ver a inclinação lateral da rampa da estrutura no canto direito e sua extensão total a esquerda a cúpula branca arredondada que lembra um disco ou concha centralizado na composição eleva-se o alto pedestal de concreto texturizado sustentando a estátua de bronze de Juscelino Kubitschek",
    "estilo_arquitetura_brasilia, a estátua A Justiça obra do escultor brasileiro Alfredo Ceschiatti localizada em frente ao Supremo Tribunal Federal a escultura é um dos símbolos mais conhecidos do Poder Judiciário brasileiro a personagem segura um objeto semelhante a um pergaminho ou livro sobre o colo simbolizando a aplicação da lei e da justiça",
    "estilo_arquitetura_brasilia, a escultura retratada é conhecida como Os Candangos um dos símbolos mais famosos de Brasília e uma homenagem aos trabalhadores que participaram da construção da capital brasileira mostra uma escultura monumental de metal escuro composta por duas figuras humanas altamente estilizadas e simétricas As figuras possuem corpos alongados pernas estreitas e cabeças pequenas sustentadas por pescoços longos Os braços elevados formam uma estrutura arqueada na parte superior da obra enquanto duas hastes metálicas verticais acompanham a composição nas extremidades O enquadramento foi feito de baixo para cima o que aumenta a sensação de imponência da escultura A superfície da obra apresenta textura irregular típica de esculturas fundidas em bronze a fotografia destaca o caráter abstrato e modernista da escultura enfatizando suas formas geométricas e sua forte simetria visual",
    "estilo_arquitetura_brasilia,O Palácio do Planalto em imagem retirada da parte direita do edificio é a sede da Presidência da República e um dos principais marcos da arquitetura modernista brasileira projetado por Oscar Niemeyer e inaugurado em 1960 durante a construção da nova capital do país Na imagem observa se uma perspectiva lateral do edifício destacando as elegantes colunas em formato curvo que parecem sustentar a estrutura com leveza O espelho dágua em primeiro plano amplia a sensação de amplitude e reflete as cores do céu ao entardecer Ao fundo aparecem as torres do Congresso Nacional reforçando a integração dos edifícios que compõem a Praça dos Três Poderes a luz dourada do pôr do sol valoriza o mármore branco da construção criando uma composição equilibrada monumental e representativa da identidade arquitetônica de Brasília",
    "estilo_arquitetura_brasilia, O Congresso Nacional é a sede do Poder Legislativo brasileiro e foi projetado por Oscar Niemeyer como parte do conjunto arquitetônico da nova capital inaugurada em 1960 A construção é composta por duas torres centrais que abrigam áreas administrativas além das cúpulas que representam o Senado Federal e a Câmara dos Deputados tornando se um dos símbolos mais reconhecidos do país Na imagem a composição frontal destaca a monumentalidade e a simetria do edifício com as torres elevando se sobre a paisagem e contrastando com as formas curvas das cúpulas O céu azul e a iluminação suave valorizam as linhas modernistas enquanto a ampla área gramada em primeiro plano reforça a sensação de espaço e imponência característica da arquitetura cívica de Brasília",
    "estilo_arquitetura_brasilia, Torre de TV de Brasília é um dos símbolos mais reconhecidos da capital federal e foi construída para atender às necessidades de transmissão de rádio e televisão durante o desenvolvimento da cidade planejada Além de sua função técnica tornou se um importante ponto turístico oferecendo vista privilegiada do Plano Piloto e do Eixo Monumental Na imagem a torre aparece em destaque absoluto ocupando o centro da composição com sua estrutura metálica elevada sobre a plataforma de observação localizada na base O enquadramento frontal e o amplo gramado em primeiro plano reforçam a sensação de altura e monumentalidade enquanto o céu azul repleto de nuvens brancas cria um contraste visual que valoriza a engenharia da construção A fotografia evidencia a integração entre funcionalidade modernista paisagem urbana e identidade arquitetônica características que fazem da Torre de TV um dos marcos mais emblemáticos de Brasília e do patrimônio urbano brasileiro",
    "estilo_arquitetura_brasilia, mostra a Catedral Metropolitana de Brasília uma obra-prima da arquitetura moderna projetada por Oscar Niemeyer cuja pedra fundamental foi lançada em 1958 e a inauguração oficial ocorreu em 1970 O monumento religioso destaca-se por sua estrutura hiperboloide marcante formada por 16 pilares curvos de concreto branco que se elevam e convergem lembrando o gesto de duas mãos se abrindo em direção ao céu que aparece bastante nublado na fotografia Entre os pilares estendem-se grandes painéis de vidro que compõem os vitrais externos do templo coroado no topo por uma cruz metálica fina sobre uma praça pavimentada com pequenos detalhes em grama na base de sua imponente fachada circular",
    "estilo_arquitetura_brasilia, mostra a Ponte Juscelino Kubitschek também conhecida como Ponte JK que cruza o Lago Paranoá em Brasília Inaugurada no final de 2002 a estrutura foi projetada pelo arquiteto Alexandre Chan e pelo engenheiro estrutural Mário Vila Verde tornando-se rapidamente um dos cartões-postais mais famosos da capital federal O monumento destaca-se por seus três monumentais arcos assimétricos de aço e concreto pintados de branco que cruzam a pista de forma diagonal simulando o movimento de uma pedra saltando sobre a superfície da água A fotografia captura os arcos sustentando o tabuleiro da ponte por meio de cabos estaiados tendo em primeiro plano uma margem verde com vegetação rasteira e arbustos sob um céu predominantemente nublado com colinas ao fundo no horizonte",
    "estilo_arquitetura_brasilia, vista aérea do Museu Nacional da República que integra o Complexo Cultural da República e foi projetado pelo renomado arquiteto Oscar Niemeyer sendo inaugurado oficialmente em 2006 para abrigar importantes exposições artísticas palestras e manifestações culturais na capital federal O monumento destaca-se por sua imensa cúpula de concreto branco em formato semiesférico com uma longa rampa externa de acesso que se projeta em direção à entrada principal enquanto o entorno revela a ampla esplanada com carros estacionados a torre cilíndrica branca à esquerda e os edifícios dos ministérios alinhados ao fundo ao lado da Catedral Metropolitana de Brasília à direita sob a imensidão do Lago Paranoá e um céu carregado de nuvens escuras no horizonte"
]

# Títulos mais curtos para a grade não ficar com os textos muito longos
titulos_curtos = [
    "1. Teatro Nacional", "2. Memorial JK", "3. A Justiça",
    "4. Os Candangos", "5. Palácio do Planalto", "6. Congresso Nacional",
    "7. Torre de TV", "8. Catedral", "9. Ponte JK", "10. Museu Nacional"
]

print(f"Gerando {len(prompts_overfitting)} imagens para o teste de memorização (overfitting)...")

imagens_geradas = []

# Gerar as imagens e guardá-las na lista
for i, prompt_txt in enumerate(prompts_overfitting):
    print(f"Gerando imagem {i+1}/10...")
    img = pipe(prompt_txt, generator=torch.Generator(device="cuda").manual_seed(seed)).images[0]
    imagens_geradas.append(img)

print("\nGeração concluída! Montando a grade compilada...")

# 2. Montar a grade 5x2 usando matplotlib
fig, axes = plt.subplots(5, 2, figsize=(12, 25))
fig.suptitle("Teste de Memorização (Overfitting) - LoRA", fontsize=18, y=0.99)

for i in range(10):
    # Calcular a linha e a coluna correta na grade
    linha = i // 2
    coluna = i % 2

    # Adicionar a imagem
    axes[linha, coluna].imshow(imagens_geradas[i])
    axes[linha, coluna].set_title(titulos_curtos[i], fontsize=14)
    axes[linha, coluna].axis('off')

# Ajustar o espaçamento para ficar bonito
plt.tight_layout(rect=[0, 0, 1, 0.98])

# 3. Salvar o arquivo compilado localmente
nome_arquivo_grade = "grade_teste_memorizacao.png"
plt.savefig(nome_arquivo_grade, bbox_inches='tight')
plt.show()

# 4. Fazer o download apenas da grade
print(f"Iniciando o download do arquivo: {nome_arquivo_grade}...")
files.download(nome_arquivo_grade)

Output hidden; open in https://colab.research.google.com to view.